# The Right to be Forgotten

## What This Is
GDPR Article 17 and CCPA grant individuals the right to have their data erased. For ML systems trained on personal data, this creates a hard technical challenge: how do you remove a data subject's influence from a trained model without retraining from scratch?

## Why It Matters
Exact unlearning — full retraining from scratch minus the forget set — is always correct but prohibitively expensive for large models. A ResNet trained on 1M images takes hours; a large language model takes weeks. The field of *machine unlearning* builds approximate alternatives that achieve similar privacy guarantees in a fraction of the cost.

## Regulatory Landscape
- **GDPR Article 17**: Right to erasure, with exceptions for freedom of expression, scientific research, legal obligations
- **CCPA**: Right to deletion for California residents
- **EU AI Act**: High-risk AI systems must support data subject rights
- **Emerging**: US federal privacy legislation proposals include similar provisions

## What Unlearning Must Achieve
A verification test: after unlearning, a membership inference attack on the forget set should perform no better than chance. The gold standard is *indistinguishability* — the unlearned model should be statistically indistinguishable from a retrained-from-scratch model.

## Exact vs Approximate Unlearning

**Exact unlearning**: Remove data point(s), retrain from scratch. Provably correct, computationally infeasible at scale.

**Approximate unlearning**: Modify the trained model to approximate what retraining would produce. Efficient, but requires verification.

Key tension: speed vs verifiability. The faster the method, the harder it is to certify that unlearning actually happened.

In [ ]:
import numpy as np
import random
from collections import defaultdict
from typing import List, Tuple, Optional

# Simulate a tiny logistic regression trained on a dataset
# We will demonstrate exact unlearning via retraining baseline

np.random.seed(42)
random.seed(42)

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def train_logistic(X, y, lr=0.1, epochs=200):
    """Simple logistic regression via gradient descent."""
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    for _ in range(epochs):
        z = X @ w + b
        pred = sigmoid(z)
        err = pred - y
        w -= lr * (X.T @ err) / n
        b -= lr * err.mean()
    return w, b

def predict(X, w, b):
    return (sigmoid(X @ w + b) > 0.5).astype(int)

def accuracy(X, y, w, b):
    return (predict(X, w, b) == y).mean()

# Generate synthetic dataset — 500 samples, 10 features
N = 500
D = 10
X_all = np.random.randn(N, D)
true_w = np.random.randn(D)
y_all = (sigmoid(X_all @ true_w) > 0.5).astype(float)

# Designate 20 samples as the 'forget set' (data subjects requesting deletion)
forget_idx = list(range(50, 70))  # indices 50-69
retain_idx = [i for i in range(N) if i not in forget_idx]

X_retain = X_all[retain_idx]
y_retain = y_all[retain_idx]
X_forget = X_all[forget_idx]
y_forget = y_all[forget_idx]

print(f'Total samples: {N}')
print(f'Retain set: {len(retain_idx)} samples')
print(f'Forget set: {len(forget_idx)} samples')


In [ ]:
# Train three models:
# 1. Original model (trained on all data)
# 2. Retrained model (exact unlearning: trained only on retain set)
# 3. We'll compare these as the baseline

print('Training original model (all data)...')
w_orig, b_orig = train_logistic(X_all, y_all)
acc_orig_retain = accuracy(X_retain, y_retain, w_orig, b_orig)
acc_orig_forget = accuracy(X_forget, y_forget, w_orig, b_orig)

print('Training retrained model (retain set only — exact unlearning)...')
w_retrain, b_retrain = train_logistic(X_retain, y_retain)
acc_retrain_retain = accuracy(X_retain, y_retain, w_retrain, b_retrain)
acc_retrain_forget = accuracy(X_forget, y_forget, w_retrain, b_retrain)

print(f'\nOriginal model:')
print(f'  Retain accuracy:  {acc_orig_retain:.3f}')
print(f'  Forget accuracy:  {acc_orig_forget:.3f}')

print(f'\nRetrained model (exact unlearning baseline):')
print(f'  Retain accuracy:  {acc_retrain_retain:.3f}')
print(f'  Forget accuracy:  {acc_retrain_forget:.3f}')

# Weight difference — how much did the model change?
weight_diff = np.linalg.norm(w_retrain - w_orig)
print(f'\nWeight L2 difference (orig vs retrained): {weight_diff:.4f}')
print('This is the target for approximate unlearning methods to match.')


In [ ]:
# Membership Inference Attack as unlearning verification
# After successful unlearning, the forget set should look like non-members

def loss_mia_score(X, y, w, b):
    """Use prediction confidence as MIA score. Higher = more 'member-like'."""
    z = X @ w + b
    probs = sigmoid(z)
    # Confidence = how sure the model is about the correct label
    correct_prob = np.where(y == 1, probs, 1 - probs)
    return correct_prob

# MIA scores: higher = model is more confident about the sample (member-like)
scores_orig_forget = loss_mia_score(X_forget, y_forget, w_orig, b_orig)
scores_orig_retain = loss_mia_score(X_retain, y_retain, w_orig, b_orig)
scores_retrain_forget = loss_mia_score(X_forget, y_forget, w_retrain, b_retrain)

print('Membership Inference Attack (MIA) Verification')
print('=' * 50)
print(f'Original model:')
print(f'  Avg confidence on retain set: {scores_orig_retain.mean():.3f}')
print(f'  Avg confidence on forget set: {scores_orig_forget.mean():.3f}')
print(f'  Gap (retain - forget):        {scores_orig_retain.mean() - scores_orig_forget.mean():.4f}')

print(f'\nRetrained model (after exact unlearning):')
print(f'  Avg confidence on retain set: {scores_orig_retain.mean():.3f}')
print(f'  Avg confidence on forget set: {scores_retrain_forget.mean():.3f}')
print(f'  Gap (retain - forget):        {scores_orig_retain.mean() - scores_retrain_forget.mean():.4f}')

print('\nGoal: approximate unlearning should match the retrained model gap.')
print('Near-zero gap on forget set = successful unlearning.')
